In [ ]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


# Sentence-level pragmatic register

A per-sentence multi-label classifier — each sentence may carry several flags simultaneously. Six classes:

| Class | Detector | Expected in this corpus |
|---|---|---|
| `imperative` | `classify_sent_mood(sent) == "imperative"` (parse-tree heuristic) | High — directive prompts |
| `directive` | overlaps any `STANCE_MARKERS["directive"]` span | High |
| `collaborative` | `M_SENT_REGISTER["collaborative"]` OR `we + modal + V` parse pattern | **Near-zero — that's a finding.** Claude Code system prompts are functional/directive, not interpersonal. |
| `permissive` | `M_SENT_REGISTER["permissive"]` OR `consider X-ing` parse pattern | Modest (`please`, `you can`, `you may`) |
| `appreciative` | `M_SENT_REGISTER["appreciative"]` | **Near-zero — that's a finding.** Same reason. |
| `configuring` | `M_SENT_REGISTER["configuring"]` OR `param-noun + setter-verb` pattern | High in tool descriptions |

A sentence with none of the six flags is `none`. The four content `*_sent_pct` values can sum >100% (intentional — multi-label).

**Why we keep the near-zero classes.** Knowing what a corpus *isn't* doing is as important as knowing what it is. A near-zero `appreciative_sent_pct` across 287 prompts is a structural fact — kept deliberately, not pruned.

The two charts below: per-category × class profile (every category × every class), and per-file outliers (top files for each attested class).


### Sentence-register per category (per-sentence pragmatic classifier)

Six per-sentence flags (multi-label): `collaborative` / `permissive` / `appreciative` / `imperative` / `directive` / `configuring`. Pcts can sum to >100% within a category — that's intentional.

The chart is structured to make the absence of `collaborative` and `appreciative` speech across all categories immediately visible: those bars stay near zero everywhere by design.

In [ ]:
"""Sentence-register per category × class — Altair grouped bar."""

sr_long = pd.DataFrame([
    {
        "category": cat,
        "class":    cls,
        "sent_pct": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_pct"],
        "sent_count": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_count"],
    }
    for cat in by_category
    for cls in SENT_REGISTER_CLASSES
])

sr_class_domain = SENT_REGISTER_CLASSES
sr_class_range  = [SR_CLASS_COLORS[c] for c in sr_class_domain]

sr_chart = (
    alt.Chart(sr_long)
    .mark_bar()
    .encode(
        x=alt.X("sent_pct:Q", title="% of sentences"),
        y=alt.Y("class:N", sort=sr_class_domain, title=None),
        color=alt.Color("class:N",
                         scale=alt.Scale(domain=sr_class_domain, range=sr_class_range),
                         legend=None),
        row=alt.Row("category:N", title=None,
                     header=alt.Header(labelAngle=0, labelAlign="left")),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("class:N"),
            alt.Tooltip("sent_pct:Q",   format=".2f", title="sent %"),
            alt.Tooltip("sent_count:Q", title="sentences"),
        ],
    )
    .properties(width=480, height=140,
                title="Sentence-register per category × class (multi-label, % of sentences)")
)
sr_chart


In [ ]:
"""Per-file outliers for sentence_register — Altair 4-panel.

Top 10 files by each ATTESTED class. Near-zero classes (collaborative,
appreciative) will surface their few outlier files; if a class has fewer
than 10 nonzero files, the panel renders only what exists.
"""

sr_per_file_df = pd.DataFrame([
    {
        "path": r["path"],
        "category": r["category"],
        **{f"{cls}_sent_pct": r["metrics"]["sentence_register"][f"{cls}_sent_pct"]
           for cls in SENT_REGISTER_CLASSES},
    }
    for r in per_file_records
])

panels = []
for cls in ["collaborative", "permissive", "appreciative", "configuring"]:
    col = f"{cls}_sent_pct"
    top = sr_per_file_df.nlargest(10, col).copy()
    top = top[top[col] > 0]   # drop zero-pct entries (relevant for the rare classes)
    panel = (
        alt.Chart(top)
        .mark_bar()
        .encode(
            x=alt.X(f"{col}:Q", title="% of sentences"),
            y=alt.Y("path:N", sort="-x", title=None,
                    axis=alt.Axis(labelLimit=300, labelFontSize=9)),
            color=alt.Color("category:N",
                             scale=alt.Scale(domain=cats,
                                             range=[CATEGORY_COLORS[c] for c in cats]),
                             legend=alt.Legend(title="Category", orient="bottom",
                                                columns=len(cats))),
            tooltip=[
                alt.Tooltip("path:N"),
                alt.Tooltip("category:N"),
                alt.Tooltip(f"{col}:Q", format=".2f", title=f"{cls} sent %"),
            ],
        )
        .properties(width=320, height=240,
                    title=f"Top by `{cls}_sent_pct` (n={len(top)})")
    )
    panels.append(panel)

(panels[0] | panels[1]) & (panels[2] | panels[3])
